In [0]:
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers "datasets>=3.4.1,<4.0.0" "huggingface-hub>=0.34.0,<1.0" trl==0.22.2 unsloth unsloth_zoo tqdm multiprocess dill xxhash regex safetensors bitsandbytes accelerate peft

Using Python 3.12.3 environment at: /local_disk0/.ephemeral_nfs/envs/pythonEnv-1fb4b6a8-6600-42b4-8ef7-351c8df1aa92
Resolved 16 packages in 33ms
⠙ Preparing packages... (0/2)
⠙ Preparing packages... (0/2)
⠙ Preparing packages... (0/2)
accelerate           ------------------------------     0 B/366.97 KiB
⠙ Preparing packages... (0/2)
accelerate           ------------------------------     0 B/366.97 KiB
⠙ Preparing packages... (0/2)
accelerate           ------------------------------ 16.00 KiB/366.97 KiB
⠙ Preparing packages... (0/2)
accelerate           ------------------------------ 32.00 KiB/366.97 KiB
⠙ Preparing packages... (0/2)
accelerate           ------------------------------ 48.00 KiB/366.97 KiB
⠙ Preparing packages... (0/2)
accelerate           ------------------------------ 48.00 KiB/366.97 KiB
⠙ Preparing packages... (0/2)
accelerate           ------------------------------ 61.14 KiB/366.97 KiB
⠙ Preparing packages... (0/2)
accelerate           ---------------------------

In [0]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [0]:
!nvidia-smi

Thu Nov 20 03:09:10 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.144.03             Driver Version: 550.144.03     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          On  |   00000000:53:00.0 Off |                    0 |
| N/A   30C    P0            116W /  700W |     530MiB /  81559MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [0]:
%restart_python

In [0]:
from datasets import load_dataset, Dataset
from unsloth.chat_templates import get_chat_template

from unsloth import FastLanguageModel
import torch

from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback
from unsloth.chat_templates import train_on_responses_only
from transformers import TextStreamer

import pandas as pd
import json

### Load model and tokenizer

In [0]:
fourbit_models = [
    "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit", # Qwen 14B 2x faster
    "unsloth/Qwen3-4B-Thinking-2507-unsloth-bnb-4bit",
    "unsloth/Qwen3-8B-unsloth-bnb-4bit",
    "unsloth/Qwen3-14B-unsloth-bnb-4bit",
    "unsloth/Qwen3-32B-unsloth-bnb-4bit",

    # 4bit dynamic quants for superior accuracy and low memory use
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    "unsloth/Phi-4",
    "unsloth/Llama-3.1-8B",
    "unsloth/Llama-3.2-3B",
    "unsloth/orpheus-3b-0.1-ft-unsloth-bnb-4bit" # [NEW] We support TTS models!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-Thinking-2507",
    max_seq_length = 4096, #2048, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "hf_...", # use one if using gated models
)

==((====))==  Unsloth 2025.11.3: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 8. Max memory: 79.096 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.51G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [0]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2025.11.3 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [0]:
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen3-thinking",
)

Test inference

In [0]:
messages = [
    {"role" : "user", "content" : "Solve (x + 2)^2 = 0."}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True, # Must add for generation
    enable_thinking = False, # Disable thinking
)

_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 450, # Increase for longer outputs!
    temperature = 0.7, top_p = 0.8, top_k = 20, # For non thinking
    streamer = TextStreamer(tokenizer, skip_prompt = False),
)

<|im_start|>user
Solve (x + 2)^2 = 0.<|im_end|>
<|im_start|>assistant
<think>
Okay, let's see. I need to solve the equation (x + 2)^2 = 0. Hmm, I remember that when you have a square equal to zero, the only solution is when the base is zero, right? Because if you square any non-zero number, it's positive, so the only way a square is zero is if the number itself is zero. 

So, let me write that down. If (x + 2)^2 = 0, then taking the square root of both sides, I get x + 2 = 0. Wait, but since it's a square, technically there's only one solution here because the square root of zero is zero. Let me confirm. 

Let's think step by step. The equation is (x + 2) squared equals zero. So, in general, if (something)^2 = 0, then something must be zero. So here, the "something" is (x + 2). So, x + 2 = 0. Solving for x, I subtract 2 from both sides: x = -2. 

Let me check if that works. If I plug x = -2 back into the original equation, I get (-2 + 2)^2 = (0)^2 = 0, which is exactly what we wanted. 

### Load dataset

In [0]:
with open('training_data_examples_all.jsonl', 'r') as f:
    training_data = [json.loads(line) for line in f]

print(f"Loaded {len(training_data)} training examples")

Loaded 5154 training examples


In [0]:
n_examples = len(training_data)

TRAIN_SIZE = 0.7
EVAL_SIZE = 0.15
TEST_SIZE = 1-TRAIN_SIZE-EVAL_SIZE

n_train = int(n_examples * TRAIN_SIZE)
n_eval = int(n_examples * EVAL_SIZE)
n_test = n_examples - n_train - n_eval

# Assign IDs to each split
train_ids = list(range(0, n_train))
eval_ids = list(range(n_train, n_train + n_eval))
test_ids = list(range(n_train + n_eval, n_examples))

train_data = [training_data[i] for i in train_ids]
eval_data = [training_data[i] for i in eval_ids]
test_data = [training_data[i] for i in test_ids]

print(f"Total examples: {n_examples}")
print(f"Training examples: {len(train_data)}")
print(f"Eval examples: {len(eval_data)}")
print(f"Test examples: {len(test_data)}")

Total examples: 5154
Training examples: 3607
Eval examples: 773
Test examples: 774


In [0]:
def preprocess_data(data):
    df = pd.DataFrame.from_dict(data)
    df["user_message"] = "Here is the research report gathered from internal / external sources. Proceed to give your verdict on the investment opportunity. \n\n" + df["report"]
    df["assistant_output"] = "<think>\n" + df["reasoning"] + "\n</think>" + df["verdict"].astype(str)
    df.drop(["report", "reasoning", "verdict"], axis=1, inplace=True)
    return df

In [0]:
train_df = preprocess_data(train_data)
eval_df = preprocess_data(eval_data)
test_df = preprocess_data(test_data)

### Format to Dataset structure

In [0]:
train_dataset = Dataset.from_pandas(train_df)
eval_dataset = Dataset.from_pandas(eval_df)
test_dataset = Dataset.from_pandas(test_df)

In [0]:
def generate_conversation(examples):
    problems  = examples["user_message"]
    solutions = examples["assistant_output"]
    conversations = []
    for problem, solution in zip(problems, solutions):
        conversations.append([
            {"role" : "user",      "content" : problem},
            {"role" : "assistant", "content" : solution},
        ])
    return { "conversations": conversations, }
  

def formatting_prompts_func(examples):
   convos = examples["conversations"]
   texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
   return { "text" : texts, }

In [0]:
train_dataset = train_dataset.map(generate_conversation, batched = True)
train_dataset = train_dataset.map(formatting_prompts_func, batched = True)

eval_dataset = eval_dataset.map(generate_conversation, batched = True)
eval_dataset = eval_dataset.map(formatting_prompts_func, batched = True)

test_dataset = test_dataset.map(generate_conversation, batched = True)
test_dataset = test_dataset.map(formatting_prompts_func, batched = True)

Map:   0%|          | 0/3607 [00:00<?, ? examples/s]

Map:   0%|          | 0/3607 [00:00<?, ? examples/s]

Map:   0%|          | 0/773 [00:00<?, ? examples/s]

Map:   0%|          | 0/773 [00:00<?, ? examples/s]

Map:   0%|          | 0/774 [00:00<?, ? examples/s]

Map:   0%|          | 0/774 [00:00<?, ? examples/s]

In [0]:
train_dataset[100]['text']

'<|im_start|>user\nHere is the research report gathered from internal / external sources. Proceed to give your verdict on the investment opportunity. \n\nDate: April 10, 2026\n\nI. Company Overview and Business Model\n\n– AquaHarvest Inc. is a U.S.-based agricultural technology startup specializing in advanced vertical farming solutions tailored for leafy greens and herbs using proprietary aeroponic systems and AI-driven environmental controls.\n– Founded in 2018, headquartered in San Jose, California.\n– Business Model:\n  • Sells modular vertical farming units to commercial growers, grocery chains, and urban agriculture startups.\n  • Provides ongoing SaaS subscription for AI-environmental monitoring and optimization.\n  • Offers consulting and installation services for farm setup and customization.\n\n– Revenue breakdown FY25:\n  • Hardware sales: 65%\n  • SaaS subscriptions: 25%\n  • Services: 10%\n\n– Scale:\n  • FY25 revenue: $85 million, up from $42 million in FY24 (102% YoY gro

### Set up trainer

In [0]:
import mlflow
from datetime import datetime

mlflow.set_experiment('/Users/kristof.rabay.cw@carlyle.com/qwen3-4b-winterfest')

<Experiment: artifact_location='dbfs:/databricks/mlflow-tracking/174334976172118', creation_time=1763608950516, experiment_id='174334976172118', last_update_time=1763608950516, lifecycle_stage='active', name='/Users/kristof.rabay.cw@carlyle.com/qwen3-4b-winterfest', tags={'mlflow.experiment.sourceName': '/Users/kristof.rabay.cw@carlyle.com/qwen3-4b-winterfest',
 'mlflow.experimentKind': 'custom_model_development',
 'mlflow.experimentType': 'MLFLOW_EXPERIMENT',
 'mlflow.ownerEmail': 'kristof.rabay.cw@carlyle.com',
 'mlflow.ownerId': '6898077869398870'}>

In [0]:
run_name = f"qwen3-4b-sft-winterfest-{datetime.now().strftime('%Y-%m-%d-%H-%M-%s')}"
print(run_name)

qwen3-4b-sft-winterfest-2025-11-20-03-23-1763609011


In [0]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset=eval_dataset,
    n_gpus = 8,
    #callbacks = [EarlyStoppingCallback(early_stopping_patience=3)],  # Stop if no improvement for 3 evals
    args = SFTConfig(
        per_device_train_batch_size = 4,#1,
        gradient_accumulation_steps = 4,
        #warmup_steps = 5,
        warmup_ratio = 0.1,
        # num_train_epochs = 1, # Set this for 1 full training run.
        # num_train_epochs = 3,
        max_steps = 250,#30,
        #max_steps = -1,
        learning_rate = 5e-5, #2e-4, #2e-6,#2e-4,
        logging_steps = 5,
        eval_steps = 25,                   # Add evaluation
        eval_strategy = "steps",           # Evaluate during training
        save_steps = 100,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear", #cosine?
        metric_for_best_model = 'eval_loss', # for early stopping
        load_best_model_at_end=True, # for early stopping
        seed = 3407,
        output_dir = "outputs",
        report_to = 'mlflow',#"none", # Use TrackIO/WandB etc
        run_name = run_name,
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=64):   0%|          | 0/3607 [00:00<?, ? examples/s]

I0000 00:00:1763609116.206489   18103 fork_posix.cc:77] Other threads are currently calling into gRPC, skipping fork() handlers
I0000 00:00:1763609116.329040   18103 fork_posix.cc:77] Other threads are currently calling into gRPC, skipping fork() handlers


Unsloth: Tokenizing ["text"] (num_proc=64):   0%|          | 0/773 [00:00<?, ? examples/s]

In [0]:
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

I0000 00:00:1763609176.767185   18103 fork_posix.cc:77] Other threads are currently calling into gRPC, skipping fork() handlers
I0000 00:00:1763609176.874645   18103 fork_posix.cc:77] Other threads are currently calling into gRPC, skipping fork() handlers
I0000 00:00:1763609177.139937   18103 fork_posix.cc:77] Other threads are currently calling into gRPC, skipping fork() handlers
I0000 00:00:1763609177.252143   18103 fork_posix.cc:77] Other threads are currently calling into gRPC, skipping fork() handlers
I0000 00:00:1763609177.534917   18103 fork_posix.cc:77] Other threads are currently calling into gRPC, skipping fork() handlers
I0000 00:00:1763609177.655338   18103 fork_posix.cc:77] Other threads are currently calling into gRPC, skipping fork() handlers


Map (num_proc=64):   0%|          | 0/3607 [00:00<?, ? examples/s]

I0000 00:00:1763609192.088591   18103 fork_posix.cc:77] Other threads are currently calling into gRPC, skipping fork() handlers
I0000 00:00:1763609192.834191   18103 fork_posix.cc:77] Other threads are currently calling into gRPC, skipping fork() handlers


Map (num_proc=64):   0%|          | 0/773 [00:00<?, ? examples/s]

In [0]:
tokenizer.decode(trainer.train_dataset[100]["input_ids"])

'<|im_start|>user\nHere is the research report gathered from internal / external sources. Proceed to give your verdict on the investment opportunity. \n\nDate: April 10, 2026\n\nI. Company Overview and Business Model\n\n– AquaHarvest Inc. is a U.S.-based agricultural technology startup specializing in advanced vertical farming solutions tailored for leafy greens and herbs using proprietary aeroponic systems and AI-driven environmental controls.\n– Founded in 2018, headquartered in San Jose, California.\n– Business Model:\n  • Sells modular vertical farming units to commercial growers, grocery chains, and urban agriculture startups.\n  • Provides ongoing SaaS subscription for AI-environmental monitoring and optimization.\n  • Offers consulting and installation services for farm setup and customization.\n\n– Revenue breakdown FY25:\n  • Hardware sales: 65%\n  • SaaS subscriptions: 25%\n  • Services: 10%\n\n– Scale:\n  • FY25 revenue: $85 million, up from $42 million in FY24 (102% YoY gro

In [0]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " ")

'                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [0]:
early_stopping_callback = EarlyStoppingCallback(
    early_stopping_patience = 3,     # How many steps we will wait if the eval loss doesn't decrease
                                     # For example the loss might increase, but decrease after 3 steps
    early_stopping_threshold = 0.0,  # Can set higher - sets how much loss should decrease by until
                                     # we consider early stopping. For eg 0.01 means if loss was
                                     # 0.02 then 0.01, we consider to early stop the run.
)
trainer.add_callback(early_stopping_callback)

### Kick off training

In [0]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA H100 80GB HBM3. Max memory = 79.096 GB.
3.955 GB of memory reserved.


In [0]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,607 | Num Epochs = 2 | Total steps = 250
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 66,060,288 of 4,088,528,384 (1.62% trained)
2025/11/20 03:27:14 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
25,1.788100,1.702271
50,1.405600,1.389706
75,1.312900,1.311063
100,1.277000,1.275373
125,1.245000,1.252673
150,1.230300,1.237399
175,1.267600,1.227291
200,1.228000,1.219875
225,1.235200,1.215927
250,1.195800,1.214328


Unsloth: Not an error, but Qwen3ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
2025/11/20 03:55:24 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2025/11/20 03:55:24 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!


In [0]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

1688.7442 seconds used for training.
28.15 minutes used for training.
Peak reserved memory = 25.369 GB.
Peak reserved memory for training = 21.414 GB.
Peak reserved memory % of max memory = 32.074 %.
Peak reserved memory for training % of max memory = 27.073 %.


### Inference

In [0]:
messages = [
    {"role" : "user", "content" : """Here is the research report gathered from internal / external sources. Proceed to give your verdict on the investment opportunity.
     
    Company is a good company, competition is a bad competition.
    """}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True, # Must add for generation
    enable_thinking = False, # Disable thinking
)

_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 2048, # Increase for longer outputs!
    temperature = 0.2, top_p = 0.8, top_k = 20, # For non thinking
    streamer = TextStreamer(tokenizer, skip_prompt = False),
)

<|im_start|>user
Here is the research report gathered from internal / external sources. Proceed to give your verdict on the investment opportunity.
     
    Company is a good company, competition is a bad competition.
    <|im_end|>
<|im_start|>assistant
<think>
Okay, looking at this company, it's described as a "good company" but the competition is a "bad competition." Hmm, that's an interesting contrast. Let me unpack this.

First, the company itself seems solid. That probably means it has strong fundamentals, good management, maybe a defensible market position, or some unique strengths. But the competition being "bad" is a bit vague. What does that mean exactly? Are they facing weak competitors who are struggling, or are they up against a strong, aggressive competitor that's making it hard for them to grow?

If the competition is weak, then the company's market position could be very strong, which would be a positive sign. But if the competition is strong and aggressive, then even 

### Save snapshot

In [0]:
model.save_pretrained("verdict_model")  # Local saving
tokenizer.save_pretrained("verdict_model")
# model.push_to_hub("your_name/lora_model", token = "...") # Online saving
# tokenizer.push_to_hub("your_name/lora_model", token = "...") # Online saving

('verdict_model/tokenizer_config.json',
 'verdict_model/special_tokens_map.json',
 'verdict_model/chat_template.jinja',
 'verdict_model/vocab.json',
 'verdict_model/merges.txt',
 'verdict_model/added_tokens.json',
 'verdict_model/tokenizer.json')

In [0]:
# Set False to True to read local model:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "verdict_model", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = 2048,
        load_in_4bit = True,
    )